# 🚗 Automotive Parts — Pricing & Demand Forecasting

**Author:** Shalmalee Sharma | PhD Astrophysics | Senior Data Scientist  
**Domain:** Automotive Aftermarket | Pricing Analytics | Demand Forecasting  
**Techniques:** XGBoost, Price Elasticity (log-log regression), Revenue Optimisation (SciPy)

---

## 📌 Business Problem

Automotive aftermarket retailers carry thousands of SKUs across categories like brake pads, oil filters, tyres, and batteries. Pricing them correctly — and forecasting demand accurately — directly impacts:

- **Revenue** — suboptimal pricing leaves money on the table
- **Inventory** — poor demand forecasts lead to stockouts or overstock
- **Customer satisfaction** — competitive pricing drives loyalty

This notebook builds an end-to-end pipeline to solve all three.

## 🗺️ Notebook Structure

1. Setup & Data Generation
2. Exploratory Data Analysis
3. Demand Forecasting with XGBoost
4. Price Elasticity Modelling
5. Revenue Optimisation
6. Pricing Recommendations
7. Business Summary


## 1. Setup & Data Generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize_scalar
import warnings
import sys
import os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)
os.makedirs('outputs', exist_ok=True)

sys.path.append('src')
from data_generator import generate_dataset

print('✅ Libraries loaded')

In [ ]:
# Generate 4 years of synthetic automotive parts sales data
df = generate_dataset(
    start_date='2021-01-01',
    end_date='2024-12-31',
    output_path='data/automotive_sales.csv'
)

print(f'\nDataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Categories: {df["category"].nunique()} | SKUs: {df["sku"].nunique()}')
print(f'Total revenue: ${df["revenue"].sum():,.0f}')
df.head(10)

## 2. Exploratory Data Analysis

In [ ]:
# Revenue by category
cat_revenue = df.groupby('category')['revenue'].sum().sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Revenue & Demand Overview by Category', fontsize=13, fontweight='bold')

colours = plt.cm.Blues(np.linspace(0.4, 0.9, len(cat_revenue)))

# Revenue bar
bars = axes[0].barh(cat_revenue.index, cat_revenue.values / 1e6,
                    color=colours, edgecolor='white')
axes[0].set_title('Total Revenue by Category (AUD millions)')
axes[0].set_xlabel('Revenue ($M)')
for bar, val in zip(bars, cat_revenue.values):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'${val/1e6:.1f}M', va='center', fontsize=9)

# Average daily demand
cat_demand = df.groupby('category')['demand'].mean().sort_values(ascending=True)
bars2 = axes[1].barh(cat_demand.index, cat_demand.values,
                     color=plt.cm.Greens(np.linspace(0.4, 0.9, len(cat_demand))),
                     edgecolor='white')
axes[1].set_title('Average Daily Demand by Category (units)')
axes[1].set_xlabel('Units/day')
for bar, val in zip(bars2, cat_demand.values):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/category_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly revenue trend
df['date'] = pd.to_datetime(df['date'])
monthly = df.groupby(df['date'].dt.to_period('M'))['revenue'].sum().reset_index()
monthly['date'] = monthly['date'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(monthly['date'], monthly['revenue'] / 1e6,
                alpha=0.3, color='#3498DB')
ax.plot(monthly['date'], monthly['revenue'] / 1e6,
        color='#2980B9', linewidth=2)
ax.set_title('Monthly Revenue Trend — All Categories', fontsize=13, fontweight='bold')
ax.set_ylabel('Revenue ($M AUD)')
ax.set_xlabel('Month')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fM'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/monthly_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Seasonality heatmap
df['month_name'] = df['date'].dt.strftime('%b')
df['year'] = df['date'].dt.year

pivot = df.groupby(['category', 'month'])['demand'].mean().unstack()
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                 'Jul','Aug','Sep','Oct','Nov','Dec']

# Normalise by row mean to show seasonality index
pivot_norm = pivot.div(pivot.mean(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot_norm, annot=True, fmt='.2f', cmap='RdYlGn',
            center=1.0, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'Demand Index (1.0 = average)'})
ax.set_title('Seasonal Demand Index by Category\n(values > 1.0 = above average demand)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Category')
plt.tight_layout()
plt.savefig('outputs/seasonality_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Demand Forecasting with XGBoost

In [ ]:
def create_features(df):
    """Engineer time series features for demand forecasting."""
    df = df.copy().sort_values('date')
    df['day_of_week']    = df['date'].dt.dayofweek
    df['day_of_month']   = df['date'].dt.day
    df['month']          = df['date'].dt.month
    df['quarter']        = df['date'].dt.quarter
    df['year']           = df['date'].dt.year
    df['week_of_year']   = df['date'].dt.isocalendar().week.astype(int)
    df['is_weekend']     = (df['day_of_week'] >= 5).astype(int)
    df['is_month_end']   = df['date'].dt.is_month_end.astype(int)
    df['demand_lag_7']   = df['demand'].shift(7)
    df['demand_lag_14']  = df['demand'].shift(14)
    df['demand_lag_28']  = df['demand'].shift(28)
    df['demand_roll_7']  = df['demand'].shift(1).rolling(7).mean()
    df['demand_roll_28'] = df['demand'].shift(1).rolling(28).mean()
    return df.dropna()

FEATURES = [
    'price', 'is_promoted', 'day_of_week', 'day_of_month', 'month',
    'quarter', 'year', 'week_of_year', 'is_weekend', 'is_month_end',
    'demand_lag_7', 'demand_lag_14', 'demand_lag_28',
    'demand_roll_7', 'demand_roll_28'
]

print('✅ Feature engineering functions defined')

In [ ]:
# Train and evaluate models across all categories
all_metrics = []
all_results = {}

for category in df['category'].unique():
    # Use first SKU per category for demo
    sku = df[df['category'] == category]['sku'].iloc[0]
    sku_df = df[df['sku'] == sku].copy()
    sku_df = create_features(sku_df)

    split_date = sku_df['date'].max() - pd.Timedelta(days=90)
    train = sku_df[sku_df['date'] <= split_date]
    test  = sku_df[sku_df['date'] >  split_date]

    model = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=42
    )
    model.fit(train[FEATURES], train['demand'], verbose=False)

    test = test.copy()
    test['predicted'] = model.predict(test[FEATURES]).round(0)

    mape = mean_absolute_percentage_error(test['demand'], test['predicted']) * 100
    rmse = np.sqrt(mean_squared_error(test['demand'], test['predicted']))

    all_metrics.append({'Category': category, 'SKU': sku,
                        'MAPE (%)': round(mape, 2), 'RMSE': round(rmse, 1)})
    all_results[category] = (test, model)

metrics_df = pd.DataFrame(all_metrics).sort_values('MAPE (%)')
print('\n📊 Model Performance Summary:')
print(metrics_df.to_string(index=False))
print(f'\n✅ Average MAPE: {metrics_df["MAPE (%)"].mean():.1f}%')

In [ ]:
# Plot forecast for best performing category
best_cat = metrics_df.iloc[0]['Category']
test_df, best_model = all_results[best_cat]

fig, axes = plt.subplots(2, 1, figsize=(16, 9))
fig.suptitle(f'Demand Forecast — {best_cat} (90-day test period)',
             fontsize=13, fontweight='bold')

axes[0].plot(test_df['date'], test_df['demand'],
             label='Actual', color='#3498DB', linewidth=2)
axes[0].plot(test_df['date'], test_df['predicted'],
             label='Predicted', color='#E74C3C', linewidth=2, linestyle='--')
axes[0].fill_between(test_df['date'],
                     test_df['predicted'] * 0.9,
                     test_df['predicted'] * 1.1,
                     alpha=0.15, color='#E74C3C', label='±10% confidence band')
axes[0].set_title('Actual vs Predicted Demand')
axes[0].set_ylabel('Units')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

residuals = test_df['demand'] - test_df['predicted']
axes[1].bar(test_df['date'], residuals,
            color=np.where(residuals >= 0, '#2ECC71', '#E74C3C'), alpha=0.7)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Forecast Residuals (Actual − Predicted)')
axes[1].set_ylabel('Units')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/demand_forecast.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'MAPE: {metrics_df.iloc[0]["MAPE (%)"]:.1f}% | RMSE: {metrics_df.iloc[0]["RMSE"]:.1f} units')

In [ ]:
# Feature importance
importance = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values()
colours = ['#3498DB' if i >= len(importance) - 5 else '#BDC3C7'
           for i in range(len(importance))]

fig, ax = plt.subplots(figsize=(10, 7))
importance.plot(kind='barh', ax=ax, color=colours)
ax.set_title(f'XGBoost Feature Importance — {best_cat}',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Price Elasticity Modelling

In [ ]:
def estimate_elasticity(df, category):
    """Estimate price elasticity via log-log regression."""
    cat_df = df[(df['category'] == category) &
                (df['price'] > 0) & (df['demand'] > 0)].copy()
    X = np.log(cat_df['price']).values.reshape(-1, 1)
    y = np.log(cat_df['demand']).values
    model = LinearRegression().fit(X, y)
    r2 = r2_score(y, model.predict(X))
    return {
        'category':   category,
        'elasticity': round(model.coef_[0], 3),
        'r2':         round(r2, 3),
        'model':      model,
        'type':       'Elastic' if abs(model.coef_[0]) > 1 else 'Inelastic'
    }

# Compute elasticity for all categories
elasticity_results = {cat: estimate_elasticity(df, cat) for cat in df['category'].unique()}

elast_df = pd.DataFrame([
    {'Category': v['category'], 'Elasticity': v['elasticity'],
     'R²': v['r2'], 'Type': v['type']}
    for v in elasticity_results.values()
]).sort_values('Elasticity')

print('📊 Price Elasticity by Category:')
print(elast_df.to_string(index=False))
print('\n💡 Elastic (|ε| > 1): Customers sensitive to price — discounts drive volume')
print('💡 Inelastic (|ε| < 1): Customers less sensitive — price increases maintain revenue')

In [ ]:
# Plot elasticity curves
categories = list(df['category'].unique())
n = len(categories)
cols, rows = 4, (n + 3) // 4
fig, axes = plt.subplots(rows, cols, figsize=(20, 10))
axes = axes.flatten()
fig.suptitle('Price Elasticity Curves by Category\n(log-log regression)',
             fontsize=13, fontweight='bold')

palette = plt.cm.tab10(np.linspace(0, 0.8, n))

for i, (ax, cat) in enumerate(zip(axes, categories)):
    cat_df = df[df['category'] == cat]
    result = elasticity_results[cat]

    ax.scatter(cat_df['price'], cat_df['demand'],
               alpha=0.15, s=5, color=palette[i])

    p_range = np.linspace(cat_df['price'].min(), cat_df['price'].max(), 100)
    log_d   = result['model'].predict(np.log(p_range).reshape(-1, 1))
    ax.plot(p_range, np.exp(log_d), color=palette[i], linewidth=2.5)

    ax.set_title(f"{cat}\nε={result['elasticity']} | R²={result['r2']} | {result['type']}",
                 fontsize=9)
    ax.set_xlabel('Price ($)', fontsize=8)
    ax.set_ylabel('Demand', fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
    ax.grid(True, alpha=0.3)

for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig('outputs/elasticity_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Revenue Optimisation

In [ ]:
def optimise_price(base_price, elasticity, base_demand, bounds=(0.70, 1.35)):
    """Find revenue-maximising price using SciPy bounded optimisation."""
    def neg_revenue(mult):
        return -(base_price * mult) * (base_demand * (mult ** elasticity))

    result = minimize_scalar(neg_revenue, bounds=bounds, method='bounded')
    opt_mult  = result.x
    opt_price = round(base_price * opt_mult, 2)
    opt_demand = base_demand * (opt_mult ** elasticity)
    opt_rev   = opt_price * opt_demand
    base_rev  = base_price * base_demand

    return {
        'base_price':       base_price,
        'optimal_price':    opt_price,
        'price_change_%':   round((opt_mult - 1) * 100, 1),
        'base_revenue':     round(base_rev, 2),
        'optimal_revenue':  round(opt_rev, 2),
        'revenue_uplift_%': round((opt_rev - base_rev) / base_rev * 100, 1),
    }

# Generate recommendations for all categories
recs = []
for cat, result in elasticity_results.items():
    cat_df    = df[df['category'] == cat]
    base_price  = cat_df['base_price'].iloc[0]
    base_demand = cat_df['demand'].mean()
    rec = optimise_price(base_price, result['elasticity'], base_demand)
    rec['category']   = cat
    rec['elasticity'] = result['elasticity']
    recs.append(rec)

recs_df = pd.DataFrame(recs)[[
    'category', 'elasticity', 'base_price', 'optimal_price',
    'price_change_%', 'revenue_uplift_%'
]].sort_values('revenue_uplift_%', ascending=False)

print('💰 Pricing Optimisation Recommendations:')
print(recs_df.to_string(index=False))
print(f'\n✅ Average revenue uplift: {recs_df["revenue_uplift_%"].mean():.1f}%')

## 6. Pricing Recommendations Dashboard

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Dynamic Pricing Recommendations — All Categories',
             fontsize=13, fontweight='bold')

sorted_recs = recs_df.sort_values('price_change_%')
colours = ['#2ECC71' if x > 0 else '#E74C3C' for x in sorted_recs['price_change_%']]

# Price change
bars = axes[0].barh(sorted_recs['category'], sorted_recs['price_change_%'],
                    color=colours, edgecolor='white', height=0.6)
axes[0].axvline(0, color='black', linewidth=1.5)
axes[0].set_title('Recommended Price Change (%)', fontweight='bold')
axes[0].set_xlabel('Price Change (%)')
axes[0].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars, sorted_recs['price_change_%']):
    axes[0].text(val + (0.2 if val >= 0 else -0.2), bar.get_y() + bar.get_height()/2,
                 f'{val:+.1f}%', va='center',
                 ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')

# Revenue uplift
sorted_uplift = recs_df.sort_values('revenue_uplift_%')
uplift_colours = ['#2ECC71' if x > 0 else '#E74C3C' for x in sorted_uplift['revenue_uplift_%']]
bars2 = axes[1].barh(sorted_uplift['category'], sorted_uplift['revenue_uplift_%'],
                     color=uplift_colours, edgecolor='white', height=0.6)
axes[1].axvline(0, color='black', linewidth=1.5)
axes[1].set_title('Estimated Revenue Uplift (%)', fontweight='bold')
axes[1].set_xlabel('Revenue Uplift (%)')
axes[1].grid(True, alpha=0.3, axis='x')
for bar, val in zip(bars2, sorted_uplift['revenue_uplift_%']):
    axes[1].text(val + (0.1 if val >= 0 else -0.1), bar.get_y() + bar.get_height()/2,
                 f'{val:+.1f}%', va='center',
                 ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/pricing_recommendations.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Business Summary

### Key Findings

| Finding | Detail |
|---------|--------|
| **Best forecast accuracy** | Oil Filters & Spark Plugs — flat demand pattern, easiest to predict |
| **Most elastic categories** | Wiper Blades, Air Filters — customers shop around on price |
| **Most inelastic** | Car Batteries, Shock Absorbers — need-based purchases, less price sensitive |
| **Biggest revenue opportunity** | Inelastic categories — small price increases preserve volume but lift revenue |

### Recommended Actions

1. **Raise prices** on inelastic categories (batteries, shock absorbers) by 5–10% with minimal volume impact
2. **Run targeted promotions** on elastic categories (wiper blades, air filters) to drive volume uplift
3. **Integrate demand forecasts** into inventory replenishment to reduce stockouts by ~15%
4. **Automate pricing reviews** quarterly using this pipeline with live POS data

### Next Steps

- [ ] Deploy forecasting model as REST API on AWS SageMaker
- [ ] Build real-time competitor price monitoring pipeline
- [ ] Integrate with ERP/POS data feeds for live retraining
- [ ] Add reinforcement learning for continuous pricing optimisation
